### Coleta de dados de Notificação de casos de Dengue

Fonte: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/csv/DENGBR26.csv.zip


In [ ]:
import cdsapi
import sys, os
import xarray as xr
import dask.dataframe as dd
from pyspark.sql import functions as F

In [ ]:
# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

In [ ]:
# https://servicodados.ibge.gov.br/api/v1/localidades/estados

ufs = {"11":"RO", "12":"AC", "13":"AM", "14":"RR", "15":"PA", "16":"AP", "17":"TO"
      ,"21":"MA", "22":"PI", "23":"CE", "24":"RN", "25":"PB", "26":"PE", "27":"AL"
      ,"28":"SE", "29":"BA", "31":"MG", "32":"ES", "33":"RJ", "35":"SP"
      ,"41":"PR", "42":"SC", "43":"RS", "50":"MS", "51":"MT", "52":"GO", "53":"DF"
      }


In [ ]:
df_dengue = \
    spark.read.csv(f"{PROJECT_PATH}\\DENGBR26.CSV"
                  ,header=True
                  ,sep=","
                  ,inferSchema=True)

In [ ]:
df_dengue.printSchema()

In [ ]:
df_dengue.select("UF", "coufinf").drop_duplicates().show(1000,False)

In [ ]:
expr = F.create_map([F.lit(x) for kv in ufs.items() for x in kv])

df_dengue = \
    df_dengue.withColumns({"MES": F.month(F.col('DT_NOTIFIC'))
                          ,"sigla_uf": expr[F.col("sg_uf_not")] })

df_dengue_cols = \
    (df_dengue
        .select("TP_NOT"
            ,"NU_ANO"
            ,"MES"
            ,F.col("NU_IDADE_N").substr(2,3).cast("int").alias("IDADE")
            ,"ID_MUNICIP"
            ,"sg_uf_not"
            ,"sigla_uf"
            ,"sg_uf"))

df_dengue_cols.createOrReplaceTempView("notif_dengue")

In [ ]:
query_dengue = \
    """Select sg_uf, nu_ano, count(1)
         from notif_dengue
        where 1=1
          -- id_municip in (110001, 110003, 110002) 
        group by all 
    """

spark.sql(query_dengue).orderBy('sig_uf').show()

In [ ]:
colunas = ['TP_NOT','ID_AGRAVO','DT_NOTIFIC','SEM_NOT','NU_ANO','SG_UF_NOT','ID_MUNICIP','ID_REGIONA','ID_UNIDADE','DT_SIN_PRI','SEM_PRI','ANO_NASC','NU_IDADE_N','CS_SEXO','CS_GESTANT','CS_RACA','CS_ESCOL_N','SG_UF','ID_MN_RESI','ID_RG_RESI','ID_PAIS','DT_INVEST','ID_OCUPA_N','FEBRE','MIALGIA','CEFALEIA','EXANTEMA','VOMITO','NAUSEA','DOR_COSTAS','CONJUNTVIT','ARTRITE','ARTRALGIA','PETEQUIA_N','LEUCOPENIA','LACO','DOR_RETRO','DIABETES','HEMATOLOG','HEPATOPAT','RENAL','HIPERTENSA','ACIDO_PEPT','AUTO_IMUNE','DT_CHIK_S1','DT_CHIK_S2','DT_PRNT','RES_CHIKS1','RES_CHIKS2','RESUL_PRNT','DT_SORO','RESUL_SORO','DT_NS1','RESUL_NS1','DT_VIRAL','RESUL_VI_N','DT_PCR','RESUL_PCR_','SOROTIPO','HISTOPA_N','IMUNOH_N','HOSPITALIZ','DT_INTERNA','UF','MUNICIPIO','TPAUTOCTO','COUFINF','COPAISINF','COMUNINF','CLASSI_FIN','CRITERIO','DOENCA_TRA','CLINC_CHIK','EVOLUCAO','DT_OBITO','DT_ENCERRA','ALRM_HIPOT','ALRM_PLAQ','ALRM_VOM','ALRM_SANG','ALRM_HEMAT','ALRM_ABDOM','ALRM_LETAR','ALRM_HEPAT','ALRM_LIQ','DT_ALRM','GRAV_PULSO','GRAV_CONV','GRAV_ENCH','GRAV_INSUF','GRAV_TAQUI','GRAV_EXTRE','GRAV_HIPOT','GRAV_HEMAT','GRAV_MELEN','GRAV_METRO','GRAV_SANG','GRAV_AST','GRAV_MIOC','GRAV_CONSC','GRAV_ORGAO','DT_GRAV','MANI_HEMOR','EPISTAXE','GENGIVO','METRO','PETEQUIAS','HEMATURA','SANGRAM','LACO_N','PLASMATICO','EVIDENCIA','PLAQ_MENOR','CON_FHD','COMPLICA','TP_SISTEMA','NDUPLIC_N','DT_DIGITA','CS_FLXRET','FLXRECEBI','MIGRADO_W']

for col in colunas:
    print(col)

In [ ]:
path_contrato = r"C:\Users\DRT90628\Downloads\urn-datacontract-edgeconsumerbigdata-hive-raw_datasus-datasus_notificacao_casos_dengue.yaml"

# id_contrato: urn:datacontract:edgeconsumerbigdata:hive:raw_datasus:datasus_notificacao_casos_dengue

In [ ]:
import yaml

# 1. Carrega o arquivo YAML
with open(path_contrato, "r", encoding="utf-8") as file:
    data = yaml.safe_load(file)

# 2. Navega até o dicionário de campos do modelo especificado
try:
    fields = data["models"]["datasus_notificacao_casos_dengue"]["fields"]
    
    # 3. Itera sobre os campos (chave: nome da coluna, valor: atributos da coluna)
    for field_name, field_info in fields.items():
        field_type = field_info.get("type")
        description = field_info.get("description")
        required = field_info.get("required")
        primary_key = field_info.get("primaryKey")
        precision = field_info.get("precision")


        print(f"{field_name};{field_type};{precision}")

        
        # # Exemplo de saída/processamento com os dados obtidos
        # print(f"Campo: {field_name}")
        # print(f"  - Tipo: {field_type}")
        # print(f"  - Obrigatório: {required}")
        # print(f"  - Chave Primária: {primary_key}")
        # print(f"  - Descrição: {description}")
        # print("-" * 50)

except KeyError as e:
    print(f"Erro ao acessar a chave no YAML: {e}")